# Clinical Trials Data Analysis

### Clinical Trial Data Analysis Using Spark SQL

# Introduction

This project explores a big data registry of clinical trials registered in the US, within the CSV file Clinicaltrial_16012025.csv. The data is from ClinicalTrials.gov and has over 500,000 records, each representing one clinical trial. The dataset includes the primary information such as trial name, type, sponsor, medical conditions under study, and trial duration (start and end dates). 

This information is for a pharmaceutical company to gain insight into the clinical trial landscape and also understand market trends. Using Spark SQL, the project involves executing a series of analytical queries on the dataset to: 
<ul>
    <li>Retrieve and rank all kinds of clinical trials by frequency.</li> 
</ul>
<ul>
  <li>Identify the top 10 most studied conditions, taking into account more than one condition reported in a record. 
</ul>
<ul>
 <li>Calculate the average trial duration (in months) for trials with valid start and completion dates. 
</ul>
Visualize the yearly trend of trials completed on Diabetes, based on the occurrence of the term "Diabetes" in the condition field and a non-null completion date. 

These task will help uncover key patterns and guide strategic decision-making based on the clinical research.

### Data Import and Preprocessing

Here is the data import and preprocessing steps taken to read the clinical trials CSV file into a DataFrame and clean it for SQL analysis. First, I uploaded the CSV file containing clinical trial data into Databricks. Using the Databricks UI, I created a table from the uploaded CSV file to determine the type of data each column contains. The schema was automatically inferred determining the data types of each column reducing errors from incorrect assumptions, with the first row of the file used as the header to define the column names. This process ensured that the data was correctly structured and ready for analysis.

In [ ]:
-- Selects all columns and all rows from the clinicaltrial table
select * from clinicaltrial

Following the import, the inferred schema and header row provided an accurate representation of the data so no additional cleaning steps were needed. The resulting table is now ready for further SQL-based analysis in the subsequent code cells.

### Details of ClinicalTrial Data

The table clinicaltrial contains several columns providing key information about clinical trials. These include the **NCT Number** (unique trial identifier), **Study Title**, and **Acronym** (if available). It also contains the **Study Status** (e.g., completed, ongoing), **Conditions** (medical conditions studied), and **Interventions** (treatments being tested). Additional details include the **Sponsor** and **Collaborators**, **Enrollment** (number of participants), and **Funder Type** (source of funding). The **Study Type** and **Study Design** describe the nature and structure of the trial, while the **Start Date** and **Completion Date** indicate the trial's timeline. These columns offer a comprehensive view of each clinical trial's details, design, and progress.

These elements provide detailed information about the trials' objectives, design, and progress, essential for further analysis.

### Structure of ClinicalTrial Data

The query DESCRIBE TABLE displays the structure of the clinicaltrial table which is essential for understanding the data types and relationships between columns, which helps in appropriate analysis and efficient querying.. Understanding the structure is crucial for planning effective data transformations.

In [ ]:
-- Displays the structure of the clinicaltrial table; column names, data types, and other metadata
DESCRIBE TABLE clinicaltrial;

The results include data types: string, which represents text; integer, which represents whole numbers; and timestamp, which represents date and time values.

This query returns the total number of patient records which helps to quickly determine the dataset size, verify data loads, audit data integrity, and provide a summary for reporting purposes.

In [ ]:
-- Returns the total number of records (rows) in the clinicaltrial table
SELECT COUNT(*) FROM clinicaltrial;

The result shows there are 522,660 rows which represents clinical trials.

###Question 1:###
List all the clinical trial types (as contained in the Type column of the data) along
with their frequency, sorting the results from most to least frequent.

In [ ]:
-- Selects each unique (non-null) study type and counts how many times it appears in the clinicaltrial table
SELECT DISTINCT `Study Type`, COUNT(*) AS Type_Frequency
FROM Clinicaltrial

-- Filters out rows where the Study Type is missing (NULL values)
WHERE `Study Type` IS NOT NULL

-- Groups the results by each Study Type so that COUNT(*) works per group
GROUP BY `Study Type`

-- Sorts the output so the most frequent Study Type appears first
ORDER BY Type_Frequency DESC;

**QUESTION 1: RESULT**

The query result list clinical study types grouped by their frequencies, ordered from the most common type, INTERVENTIONAL (399,888), to the least common type, EXPANDED ACCESS (966) providing a clear overview of the distribution of study types. This distribution indicates the prevalence of interventional studies, which are designed to measure the effect of interventions or treatments on health outcomes. Expanded access studies, providing patients with access to investigational treatment outside trials, occur much less often. This data emphasizes the focus on rigorous testing and verification of new treatments using interventional studies while acknowledging the place of expanded access for unmet medical needs in patients.

###Question 2:###
List the top 10 conditions along with their frequency (note, that the Condition column
can contain multiple conditions in each row, so you will need to separate these
out and count each occurrence separately).

In [ ]:
-- Selects the top 10 most frequently mentioned conditions in the clinicaltrial table
SELECT Conditions, COUNT(*) AS COND_Frequency
FROM (
    -- Splits the comma-separated Conditions field into individual condition values (one per row)
    -- and removes rows where Conditions is NULL
    SELECT EXPLODE(SPLIT(Conditions, '\\|')) AS Conditions
    FROM clinicaltrial
    WHERE Conditions IS NOT NULL
) t

-- Groups the individual condition values so we can count how many times each appears
GROUP BY t.Conditions

-- Sorts the condition list in descending order of frequency (most common first)
ORDER BY COND_Frequency DESC

-- Limits the result to the top 10 most frequently occurring conditions
LIMIT 10;

**QUESTION 2: RESULT**

The query result shows the top Ten(10) clinical conditions and their frequency from the most frequent 10309 (Healthy) to the least frequent health condition 3529 (Cancer). Breast Cancer and Obesity are the most prevalent conditions, highlighting areas that may require more focused healthcare interventions while conditions like Stroke, Hypertension, and Depression still represent a significant health concern. Cancer conditions, when combined, represent a significant portion of the overall health conditions, highlighting the importance of cancer awareness and treatment. These trends can help understand the distribution of health conditions and prioritizing healthcare resources and interventions.

### Question 3:
Calculate the mean clinical trial length in months, for studies with an end date.

In [ ]:
-- Calculates the average duration (in months) of clinical trials that have a recorded completion date
SELECT 
  ROUND(AVG(month_diff), 0) AS Mean_Clinical_Trial_Length
FROM (
  -- Subquery: Calculates the month difference between the start and completion date for each trial
  SELECT 
    `Start Date`, 
    `Completion Date`, 
    DATEDIFF(MONTH, `Start Date`, `Completion Date`) AS month_diff
  FROM clinicaltrial
  -- Filters out trials that don't have a completion date (to avoid errors in DATEDIFF)
  WHERE `Completion Date` IS NOT NULL
) AS trial_lengths;

**QUESTION 3: RESULT**

The query calculates the average length of clinical trials in months for studies with a completion date. The mean trial length, rounded to the nearest whole number, is 35 months. These results typically reflect the various phases of research, patient recruitment, and regulatory compliance, highlighting the commitment to rigorous scientific standards and the significant contributions of trial participants. While the average duration of the clinical trials above is around 35 months, this can vary widely depending on the study's complexity and phase.

### Question 4:
From the studies with a non-null completion date and a status of ‘Completed’ in
the Study Status, calculate how many of these related to Diabetes each year.
Display the trend over time in an appropriate visualisation. (For this you can
assume all relevant studies will contain an exact match for ‘Diabetes’ or ‘diabetes’
in the Conditions column.)

In [ ]:
-- Selects the year of the trial's completion and counts the number of completed trials related to diabetes for each year
SELECT YEAR(cl.`Completion Date`) AS Year, 
       COUNT(*) AS Diabetese_Count
FROM clinicaltrial cl
-- Filters for only completed studies
WHERE cl.`Study Status` = "COMPLETED"
  -- Ensures the trial is related to diabetes by checking if the Conditions column contains the word "diabetes" (case-insensitive)
  AND (LOWER(cl.Conditions) LIKE '%diabetes%')
  -- Excludes trials without a recorded completion date to avoid NULL values
  AND cl.`Completion Date` IS NOT NULL
-- Groups the results by year of completion
GROUP BY Year
-- Orders the output chronologically by year
ORDER BY Year;

**Question 4:**

The query results show a fluctuating trend in the number of completed diabetes-related studies over time. Initially, there were fewer studies, but there was a notable increase in the early 2000s. This surge suggests a growing interest or focus on diabetes research during that period. After this peak, the number of studies stabilized, indicating a consistent level of research activity in subsequent years.

#Conclusion

The clinical trial data analysis shows a number of significant points. The distribution of clinical study types shows a focus on testing new treatments and interventions. The prevalence of clinical conditions shows a high range of studied conditions, with a focus of significant magnitude placed on health maintenance and treatment of severe disease. The average length of clinical trials provides insight into the typical time period for clinical research. The trend of diabetes research varies with time, showing a rise in interest in diabetes research at certain intervals and its stabilization later on. Such observations present a picture of the clinical trial scene that is beneficial to be familiar with the status of the latest clinical research.

In the future, we can assume that the focus on interventional studies will continue, driven by the need for new treatments. The list of conditions under study will likely expand, with new medical conditions. Clinical trial durations may decrease with technology gains and efficiencies. Diabetes research will remain a high-priority area with potential expansion in studies due to rising global prevalence. Such predictions can guide future research requirements and resource allocation.

In [ ]:
# %sql
WITH q AS (SELECT YEAR(cl.`Completion Date`) AS Year, 
       COUNT(*) AS Diabetese_Count
FROM clinicaltrial cl

WHERE cl.`Study Status` = "COMPLETED"
  
  AND (LOWER(cl.Conditions) LIKE '%diabetes%')
  
  AND cl.`Completion Date` IS NOT NULL

GROUP BY Year

ORDER BY Year) SELECT `Year`,SUM(`Diabetese_Count`) `column_b046668f42` FROM q GROUP BY `Year`